# NYC 311 lab notebook

Runs the same stages as `python run_all.py`, one cell per stage, so you can read each report block as it appears.
Each stage prints a block between `===== BEGIN ... =====` and `===== END ... =====`.

**Cluster run #1** = cells 1-5: find the code, install (first time only), settings, ingest + discovery, download.

## 1. Find the code
Jupyter usually starts in the notebook's folder. If it doesn't, set `CODE` to the `nyc311_lab` folder (the one that holds `run_all.py`).

In [ ]:
import os
import sys
from pathlib import Path

CODE = None  # e.g. "/gpfs/home/<netid>/nyc311_lab"; None = this folder or a parent of it
start = Path(os.path.expanduser(CODE or os.environ.get("NYC311_CODE") or os.getcwd())).resolve()
code = next((p for p in [start, *start.parents] if (p / "run_all.py").is_file() and (p / "nyc311").is_dir()), None)
if code is None:
    raise FileNotFoundError(f"no nyc311_lab folder at or above {start}: set CODE to the folder that holds run_all.py")
os.chdir(code)
if str(code) not in sys.path:
    sys.path.insert(0, str(code))
print("code folder:", code)
%load_ext autoreload
%autoreload 2

## 2. Install packages (first time only)
Installs into your home folder (`--user`; the cluster's shared Python isn't writable). Then **Kernel > Restart**,
run cell 1 again, skip this cell, and continue at 3. (Locally, use uv instead; see the README.)

In [ ]:
%pip install --user -q -r requirements.txt

## 3. Settings
`WORK` holds the large Parquet copies: put it on persistent storage with room for about half the JSON size
(~17 GB for the full 33 GB pull) and **reuse the same folder every run**. Later runs only convert new files, and a
run that dies halfway picks up where it stopped. Use absolute paths on the cluster.

In [ ]:
DATA = "data/sample/311_batch_*.json"  # cluster: "/gpfs/project/esai_2026/data/311/311_batch_*.json"
WORK = "work"  # cluster: "/gpfs/home/<netid>/esai_311_work" (the same folder every run)
OUT = "outputs"  # small results (reports, aggregates, figures); zipped for download
THREADS = None  # None = SLURM_CPUS_PER_TASK (4 on our job)
MEMORY_LIMIT = None  # None = 70% of the job's SLURM memory (22GB of 32 GB); or e.g. "20GB"

try:
    import duckdb
    import pandas
    import pyarrow
except ImportError as e:
    raise ImportError(f"{e.name} is not installed for this kernel: run cell 2, then Kernel > Restart, "
                      "then cell 1, then this cell") from None
print(f"duckdb {duckdb.__version__} | pandas {pandas.__version__} | pyarrow {pyarrow.__version__}")

from nyc311 import bundle, config, env
from nyc311.config import Config
from run_all import run_stage

args = ["--data", os.path.expanduser(DATA), "--work", os.path.expanduser(WORK), "--out", os.path.expanduser(OUT)]
if THREADS:
    args += ["--threads", str(THREADS)]
if MEMORY_LIMIT:
    args += ["--memory-limit", MEMORY_LIMIT]
cfg = Config.from_args(args)
env.run(cfg);

## 4. Ingest + discovery (cluster run #1)
Ingest checks every file and converts JSON to Parquet (about 3 minutes for 2,824 files on 4 CPUs; files already in
`WORK` are skipped). Discovery prints blocks `02a`-`02f` (about 3-5 minutes on 28M rows).

In [ ]:
for stage in ["ingest", "discover"]:
    run_stage(stage, cfg)

## 5. Download
Download the zip this prints (well under 50 MB) and unzip it into `cluster_outputs/runN/` locally.

In [ ]:
bundle.zip_outputs(cfg)

## Later runs: cleaning, questions 1-11 and the report
Cleaning writes the typed, flagged Parquet; each question module prints its report block and writes charts to
`outputs/figs/`; the last stage writes `outputs/report.html`.

In [ ]:
run_stage("clean", cfg)

In [ ]:
for stage in config.STAGES[config.STAGES.index("clean") + 1:]:  # q01 ... q11, then report
    run_stage(stage, cfg)

## Download
Download the zip this prints **and** `outputs/report.html` (it is also inside the zip). Unzip into
`cluster_outputs/runN/` locally.

In [ ]:
bundle.zip_outputs(cfg)